# Weather Prediction Project: Notebook 2. Data Cleaning, Preprocessing, and Feature Engineering

This notebook transforms the audited weather archive into a modeling table that is both analytically stronger and easier to defend. The objective is not to create as many variables as possible, but to show how careful cleaning, temporal discipline, and targeted feature engineering can turn raw daily observations into a credible probability-of-rain dataset.

### How the preprocessing work was merged

The final preprocessing path combines the reproducible aligned pipeline with the broader feature-screening work from the additional exploratory notebooks. That gives the project one clean modeling table while still preserving the lessons from the scaled all-feature experiments.

| Integrated decision | Evidence carried forward |
| --- | --- |
| Use chronological train, validation, and test splits. | The supplementary preprocessing work correctly flagged random splitting as leakage for a forecasting task. |
| Keep station-level time order visible. | The supplementary preprocessing work reinforced a chronological forecasting view of the station network, so later observations are evaluated as future weather rather than shuffled rows. |
| Enrich the table with geographic context. | Koppen climate zone, NCC zone, rainfall zone, latitude, longitude, and elevation were kept as real weather context rather than decorative metadata. |
| Compare domain features with statistical filters. | Chi-square, mutual information, ANOVA, variance filtering, MAD, Cramer's V, and sequential selection were used as complementary checks. |
| Maintain both raw/tree-friendly and scaled/distance-friendly representations where needed. | Boosted trees, SVM-style models, KNN, and neural networks do not need the same preprocessing surface. |
| Standardize feature naming in the project narrative. | Equivalent names such as `dew_point_spread_3pm` and `dewpoint_spread_3pm` are treated as the same meteorological signal. |

The result is intentionally conservative: the final project does not pretend that every branch used identical preprocessing, but it does make the final lineage clear and keeps study-specific evidence in the correct modeling role.

<!-- INTEGRATED_PREPROCESSING_MERGE -->
### Integrated feature-engineering decisions carried forward

The merged preprocessing stage carries forward several concrete safeguards from the additional preprocessing work.

| Integrated preprocessing safeguard | Final implementation effect |
| --- | --- |
| Preserve chronological forecasting order instead of mixing time randomly. | Every later comparison keeps the forecasting direction intact: earlier observations train the model and later observations test it. |
| Preserve rare climate coverage instead of trimming inconvenient stations. | The alpine regime remains visible, so the project does not quietly delete hard-weather contexts just to simplify preprocessing. |
| Do not force one imputation rule onto every variable. | Stable fields, cloud fields, sunshine / evaporation, and directional variables stay under different handling logic. |
| Use zone-aware support only where climate coherence is visible. | Pressure-style variables can borrow more structure than cloud-style variables, which remain noisier. |
| Treat weakly supported cloud recovery conservatively. | Median-style filling is paired with explicit missingness indicators instead of pretending the recovered values are fully trustworthy. |
| Keep both a reproducible aligned table and a broader all-feature study table. | The aligned table supports fair baseline comparison, while the broader table supports boosted-tree, calibration, and deep-learning studies. |
| Scale only where the model family needs it. | KNN, SVM, and neural experiments use scaled surfaces, while tree-based experiments can stay closer to raw weather magnitudes and missingness patterns. |

In other words, the final preprocessing notebook is doing two jobs at once: it keeps one clean reproducible backbone for the final workflow, and it preserves the broader preprocessing lessons that later explain why XGBoost, calibration, and the richer hybrid winner were all worth testing.

<!-- INTEGRATED_PREPROCESS_MAP -->
### Model families and their preprocessing surfaces

One reason the merged workflow needs careful reading is that not every model family was built on the same data surface. That is not a flaw; it is part of the project design.

| Model family | Main preprocessing / feature-engineering surface | Why that surface fits the model |
| --- | --- | --- |
| Logistic Regression, Random Forest, aligned XGBoost | aligned top-25 feature table | Keeps the baseline comparison fair and interpretable on one shared tabular setup. |
| KNN and Linear SVM | scaled all-feature table | Distance- and margin-based models need standardized feature magnitudes to behave sensibly. |
| Random Forest, Decision Tree, XGBoost, LightGBM | broader all-feature weather table with climate context | Tree-based models can exploit nonlinear interactions, missingness patterns, and rich geographic context without the same scaling needs. |
| Calibrated XGBoost and calibrated LightGBM | broader all-feature boosted-tree study with threshold search and isotonic calibration | This surface was used to test whether stronger probability handling changed the final decision story. |
| Nearest Centroid and TPOT side studies | selected-feature scaled table | These side experiments tested whether a narrower feature set or automated pipeline search could stay competitive. |
| Dense / Wide & Deep / ResNet static deep models | scaled high-dimensional table | Neural models benefit from normalized feature ranges and a broader engineered input space. |
| CatBoost final winner | hybrid-plus-core winner representation | CatBoost can absorb mixed feature types and the richer hybrid weather context cleanly. |
| LSTM, GRU, Attention, CNN | rolling 68-driver sequence views | Sequence models need explicitly ordered temporal windows rather than one-row static tables. |

This map is what keeps the merged project coherent. The notebooks do not pretend that every family used identical preprocessing. Instead, they show which preprocessing surface was appropriate for which family, and the ranking notebook later makes the comparison rules explicit.

<!-- CANONICAL_FEATURE_LANGUAGE -->
### Canonical feature language used from notebooks 2 to 6

Because the merged work came from more than one source notebook, a few labels refer to the same meteorological idea under slightly different names. The final project keeps one canonical language from this point onward.

| Canonical final-project name | Earlier or alternate label | Meaning in practice |
| --- | --- | --- |
| `dewpoint_spread_3pm` | `dew_point_spread_3pm` | Afternoon dew-point spread; a key dryness-versus-rain signal. |
| `pressure_day_diff` | `Pressure_diff` | Intraday pressure change from morning to afternoon. |
| `temp_day_diff` | `Temp_diff` | Intraday temperature rise or fall across the day. |
| `humidity_day_diff` | `Humidity_diff` | Intraday humidity change from morning to afternoon. |
| `year_cycle_sin`, `year_cycle_cos` | `Month_sin`, `Month_cos` | Circular seasonal encoding carried forward into model-ready tables. |
| climate context | `koppen_zone`, `ncc_zone`, and `rainfall_zone` views | Different but complementary descriptions of Australian weather regimes. |
| 68-feature hybrid-plus-core winner representation | the saved winner-package base feature set | Canonical final CatBoost driver set used in notebooks 4, 5, and 6. |
| 83-feature geo-context extension | geo-context challenger | Controlled extension of the 68-feature base, not a new unrelated feature world. |

This section does not pretend that every historical code cell was written with the same variable names. It does make the narrative consistent: later notebooks refer back to these canonical names and surfaces so the merged project reads as one workflow instead of parallel naming branches.

## Step 1: Preserve a reproducible preprocessing backbone

The preprocessing pipeline is anchored in the shared project helpers so that the data preparation logic remains identical across notebooks, experiments, and the Streamlit application. That choice matters scientifically: when the same transformation path is reused end to end, any change in performance can be attributed to feature design or model choice rather than to hidden differences in data handling.

In [1]:
import sys
from pathlib import Path

import pandas as pd

possible_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next(
    (
        path
        for path in possible_roots
        if (path / "models").exists() and (path / "notebooks").exists()
    ),
    Path.cwd().resolve(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.features.feature_pipeline import build_features_pipeline
from src.models.ines_feature_modeling_aligned import (
    TARGET,
    TIME_COL,
    BASE_FEATURES,
    EXPANDED_FEATURES,
    add_calendar_parts,
    chronological_split,
    make_model_dataset,
    rank_raw_features,
    select_top_features,
    split_xy,
)


## Step 2: Start from the original weather archive

The workflow begins with the raw `weatherAUS.csv` file to preserve a transparent lineage from source data to final prediction. Starting from the untouched archive is important here because the later modeling claims depend on showing that improvements came from better cleaning and feature construction, not from silently replacing the dataset.

In [2]:
RAW_DATA_PATH = Path('../data/raw/weatherAUS.csv')

print('Loading from:', RAW_DATA_PATH.resolve())
print('Exists:', RAW_DATA_PATH.exists())

df_raw = pd.read_csv(RAW_DATA_PATH)
df_raw.head()


Loading from: C:\Users\user 1\Desktop\feb26_bds_int_weather\data\raw\weatherAUS.csv
Exists: True


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


**Interpretation:** The preview confirms that the notebook is still working with the original daily observations, including the same weather variables and the same missingness patterns identified during the audit. This is the right starting point because it means the next steps can be interpreted as genuine preprocessing decisions rather than as unexplained changes in the source material.

### From Audit to Preparation

The audit established four constraints that now guide preprocessing: the target must be known, temporal order must be preserved, missingness may itself carry information, and spatial context cannot be ignored in a country as climatically diverse as Australia. This notebook therefore moves from description to construction: it cleans the table, engineers interpretable weather signals, and builds a shortlist that is strong enough for model comparison without becoming arbitrary.

## Step 3: Build a clean supervised base table

Before engineering new predictors, the dataset has to be made trustworthy. Rows with unknown `RainTomorrow` values cannot train a classifier, duplicated records would artificially inflate confidence, and irrelevant columns add noise without adding signal. This step therefore defines the supervised base table on which every later comparison depends.

In [3]:
original_cols = [
    'Date', 'Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine',
    'WindGustDir', 'WindGustSpeed', 'WindDir9am', 'WindDir3pm', 'WindSpeed9am',
    'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm',
    'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'RainToday', 'RainTomorrow'
]

df_prep = df_raw[original_cols].copy()
df_prep['Date'] = pd.to_datetime(df_prep['Date'])
raw_rows = len(df_prep)
missing_target_rows = int(df_prep['RainTomorrow'].isna().sum())
duplicate_rows = int(df_prep.duplicated().sum())

df_prep = df_prep[df_prep['RainTomorrow'].notna()].drop_duplicates().copy()

prep_summary = pd.DataFrame([
    ['Raw rows kept from the original archive', raw_rows],
    ['Rows removed because RainTomorrow is missing', missing_target_rows],
    ['Exact duplicate rows removed', duplicate_rows],
    ['Rows retained for supervised modelling', len(df_prep)],
    ['Columns retained', len(df_prep.columns)],
], columns=['preprocessing_step', 'value'])

display(prep_summary)
display(df_prep.head())


,preprocessing_step,value
0,Raw rows kept from the original archive,145460
1,Rows removed because RainTomorrow is missing,3267
2,Exact duplicate rows removed,0
3,Rows retained for supervised modelling,142193
4,Columns retained,23


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


**Interpretation:** After removing unusable targets and duplicate rows, the project still retains a large modeling base, which is a good sign for statistical stability. The key point is not only the remaining sample size, but also the fact that the cleaned table still reflects the original variability of Australian weather rather than a heavily filtered subset.

## Step 4: Engineer weather signals that reflect rainfall dynamics

Rainfall is rarely driven by a single raw measurement. It is usually the result of interacting atmospheric conditions such as humidity, pressure change, cloud cover, temperature contrast, and recent rainfall history. The feature-engineering step therefore translates individual observations into composite signals that better reflect how rain risk builds over time and across locations.

In [4]:
df_feat = build_features_pipeline(
    df_prep,
    profile='aligned',
    use_geo=True,
    use_koppen=True,
    use_seasonal_rainfall=True,
    use_temperature_humidity=True,
)
df_feat = add_calendar_parts(df_feat)

engineered_columns = [column for column in df_feat.columns if column not in df_prep.columns]
engineered_preview = pd.DataFrame({
    'engineered_feature': engineered_columns[:40]
})

feature_build_summary = pd.DataFrame([
    ['Rows after feature engineering', len(df_feat)],
    ['Columns after feature engineering', len(df_feat.columns)],
    ['New engineered columns added', len(engineered_columns)],
], columns=['feature_build_step', 'value'])

display(feature_build_summary)
display(engineered_preview)
display(df_feat.head())


,feature_build_step,value
0,Rows after feature engineering,142193
1,Columns after feature engineering,94
2,New engineered columns added,94


,engineered_feature
0,date
1,location
2,min_temp
3,max_temp
4,rainfall
5,evaporation
6,sunshine
7,wind_gust_dir
8,wind_gust_speed
9,wind_dir_9am


,date,location,min_temp,max_temp,rainfall,evaporation,sunshine,wind_gust_dir,wind_gust_speed,wind_dir_9am,...,humidity_3pm_roll7_mean,pressure_3pm_roll3_mean,pressure_3pm_roll7_mean,temp_3pm_roll3_mean,temp_3pm_roll7_mean,wind_gust_speed_roll3_mean,wind_gust_speed_roll7_mean,month,year,day
0,2008-07-01,Adelaide,8.8,15.7,5.0,1.6,2.6,NW,48.0,SW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,2008,1
1,2008-07-02,Adelaide,12.7,15.8,0.8,1.4,7.8,SW,35.0,SSW,...,67.000000,1017.700000,1017.700000,14.900000,14.900000,48.000000,48.000000,7,2008,2
2,2008-07-03,Adelaide,6.2,15.1,0.0,1.8,2.1,W,20.0,NNE,...,59.500000,1020.150000,1020.150000,15.200000,15.200000,41.500000,41.500000,7,2008,3
3,2008-07-04,Adelaide,5.3,15.9,0.0,1.4,8.0,NNE,30.0,NNE,...,58.333333,1022.266667,1022.266667,14.766667,14.766667,34.333333,34.333333,7,2008,4
4,2008-07-06,Adelaide,11.3,15.7,NaN,NaN,1.5,NNW,52.0,NNE,...,55.250000,1024.900000,1023.100000,14.900000,14.900000,28.333333,33.250000,7,2008,6


**Interpretation:** The expanded feature table shows that alignment was useful because it added structure, not just volume. The increase in columns means the dataset now captures more of the temporal, spatial, and regime-level context behind rainfall formation, which is exactly the kind of enrichment we would expect to help a probabilistic classifier.

## Step 5: Define candidate feature pools before selection

At this stage the project deliberately keeps more than one candidate feature pool. A narrower baseline remains valuable because it preserves interpretability, while the expanded aligned pool tests whether geographic and climate-aware additions contribute information beyond the core meteorological variables. Keeping both is important because feature selection compares credible alternatives instead of validating a single pre-decided list.

In [5]:
manual_base_features = [feature for feature in BASE_FEATURES if feature in df_feat.columns]
optional_geo_features = [
    'lat', 'lon', 'elevation', 'ncc_zone', 'koppen_zone',
    'seasonal_rainfall_zone', 'temperature_humidity_zone', 'land_strip'
]
expanded_candidate_features = list(dict.fromkeys([
    *[feature for feature in EXPANDED_FEATURES if feature in df_feat.columns],
    *[feature for feature in optional_geo_features if feature in df_feat.columns],
]))

feature_pool_summary = pd.DataFrame([
    ['Manual base feature count', len(manual_base_features)],
    ['Expanded candidate feature count', len(expanded_candidate_features)],
    ['Optional geo-climate features available', len([feature for feature in optional_geo_features if feature in df_feat.columns])],
], columns=['feature_pool', 'value'])

optional_feature_presence = pd.DataFrame({
    'optional_geo_climate_feature': optional_geo_features,
    'available_in_table': [feature in df_feat.columns for feature in optional_geo_features],
})

display(feature_pool_summary)
display(optional_feature_presence)
display(pd.DataFrame({'expanded_candidate_feature': expanded_candidate_features}))


,feature_pool,value
0,Manual base feature count,39
1,Expanded candidate feature count,84
2,Optional geo-climate features available,8


,optional_geo_climate_feature,available_in_table
0,lat,True
1,lon,True
2,elevation,True
3,ncc_zone,True
4,koppen_zone,True
5,seasonal_rainfall_zone,True
6,temperature_humidity_zone,True
7,land_strip,True


,expanded_candidate_feature
0,location
1,ncc_zone
2,rain_today
3,max_temp
4,rainfall
...,...
79,pressure_9am_missing
80,pressure_3pm_missing
81,koppen_zone
82,seasonal_rainfall_zone


**Interpretation:** The two candidate pools reveal the central tension of the project. The baseline remains compact and meteorologically intuitive, but the expanded pool introduces location-aware structure that may matter in a continent with pronounced regional climates. At this point the question is no longer whether these new variables exist, but whether they earn their place when the selection is done properly.

## Step 6: Rank features on the training period only

For the aligned screening table, feature ranking is carried out on the pre-test training period only. That keeps the final test block out of shortlist construction and preserves the main chronological boundary of the project. It does not create a fully nested estimate for the later notebook 3 validation fold, because notebook 3 subdivides this saved pre-test period again into train and validation partitions. The ranking step should therefore be read as chronologically honest screening relative to the final holdout, not as the last word on aligned validation performance.

In [6]:
df_model_candidates = make_model_dataset(
    df_feat,
    expanded_candidate_features,
    keep_date=True,
)

train_df, test_df = chronological_split(df_model_candidates, test_size=0.20)
X_train, X_test, y_train, y_test = split_xy(train_df, test_df)

X_train_model = X_train.drop(columns=[TIME_COL], errors='ignore')
feature_ranking = rank_raw_features(X_train_model, y_train)
selected_top_25 = select_top_features(X_train_model, y_train, top_n=25)

split_summary = pd.DataFrame([
    ['Training rows', len(train_df)],
    ['Test rows', len(test_df)],
    ['Training period start', train_df[TIME_COL].min()],
    ['Training period end', train_df[TIME_COL].max()],
    ['Test period start', test_df[TIME_COL].min()],
    ['Test period end', test_df[TIME_COL].max()],
    ['Features entering the ranker', X_train_model.shape[1]],
    ['Top features kept for the shortlist', len(selected_top_25)],
], columns=['split_or_selection_step', 'value'])

display(split_summary)
display(pd.DataFrame({'selected_top_25_feature': selected_top_25}))


,split_or_selection_step,value
0,Training rows,113754
1,Test rows,28439
2,Training period start,2007-11-01 00:00:00
3,Training period end,2015-11-10 00:00:00
4,Test period start,2015-11-10 00:00:00
5,Test period end,2017-06-25 00:00:00
6,Features entering the ranker,84
7,Top features kept for the shortlist,25


,selected_top_25_feature
0,humidity_3pm
1,location
2,moisture_stability_3pm
3,rain_today
4,dew_point_spread_3pm
5,seasonal_rainfall_zone
6,koppen_zone
7,land_strip
8,sunshine
9,wind_gust_speed


**Interpretation:** The split confirms that the aligned shortlist was fixed without using the final test block. The appearance of spatial variables among the strongest candidates suggests that the aligned additions are not decorative; they are contributing signal even before the classifier is trained. At the same time, the saved top-25 list should be read as an exploratory screening surface, because notebook 3 later reuses part of this pre-test period as validation data.

## Step 7: Examine the strongest signals before modeling

Once the ranking is available, it becomes possible to ask a more substantive question: which variables are repeatedly carrying the prediction story? This inspection helps separate robust meteorological drivers from features that only appear useful because they are correlated with stronger variables nearby.

In [7]:
feature_ranking_view = feature_ranking.head(20).copy().round(4)
feature_ranking_view.insert(0, 'rank', range(1, len(feature_ranking_view) + 1))
display(feature_ranking_view)


,rank,feature,importance,importance_share
0,1,humidity_3pm,0.2039,0.2039
1,2,location,0.1502,0.1502
2,3,moisture_stability_3pm,0.0774,0.0774
3,4,rain_today,0.0415,0.0415
4,5,dew_point_spread_3pm,0.0372,0.0372
5,6,seasonal_rainfall_zone,0.0356,0.0356
6,7,koppen_zone,0.0249,0.0249
7,8,land_strip,0.0248,0.0248
8,9,sunshine,0.0223,0.0223
9,10,wind_gust_speed,0.0190,0.0190


**Interpretation:** The top of the ranking still belongs to familiar weather indicators such as humidity, moisture balance, and location. That continuity is reassuring because the aligned preprocessing did not distort the physical story of rainfall. Instead, it sharpened it by allowing selected geographic variables to appear alongside the established atmospheric drivers.

## Step 8: Compare domain judgment with data-driven selection

A good shortlist should be defensible both statistically and scientifically. Comparing the manual list with the ranked selection makes that tension visible: it shows where prior weather knowledge was already pointing in the right direction and where the data argues for keeping additional spatial or contextual features.

In [8]:
manual_only = [feature for feature in manual_base_features if feature not in selected_top_25]
selected_only = [feature for feature in selected_top_25 if feature not in manual_base_features]
shared_features = [feature for feature in selected_top_25 if feature in manual_base_features]

feature_summary = pd.DataFrame({
    'manual_base_feature': pd.Series(manual_base_features),
    'selected_top_25_feature': pd.Series(selected_top_25),
})

feature_overlap_summary = pd.DataFrame([
    ['Shared features between the manual and selected lists', len(shared_features)],
    ['Features kept only in the manual base list', len(manual_only)],
    ['Features kept only in the selected top-25 list', len(selected_only)],
], columns=['comparison', 'value'])

display(feature_summary)
display(feature_overlap_summary)
display(pd.DataFrame({'manual_only_feature': pd.Series(manual_only), 'selected_only_feature': pd.Series(selected_only)}))


,manual_base_feature,selected_top_25_feature
0,location,humidity_3pm
1,ncc_zone,location
2,rain_today,moisture_stability_3pm
3,max_temp,rain_today
4,rainfall,dew_point_spread_3pm
5,sunshine,seasonal_rainfall_zone
6,wind_gust_speed,koppen_zone
7,humidity_3pm,land_strip
8,pressure_3pm,sunshine
9,cloud_9am,wind_gust_speed


,comparison,value
0,Shared features between the manual and selecte...,11
1,Features kept only in the manual base list,28
2,Features kept only in the selected top-25 list,14


,manual_only_feature,selected_only_feature
0,ncc_zone,moisture_stability_3pm
1,max_temp,seasonal_rainfall_zone
2,cloud_9am,koppen_zone
3,temp_3pm,land_strip
4,humidity_day_diff,pressure_humidity_3pm_ratio
5,pressure_day_diff,temperature_humidity_zone
6,temp_day_diff,rainfall_missing
7,wind_speed_day_diff,cloud_humidity_3pm_interaction
8,cloud_day_diff,humidity_3pm_missing
9,humidity_overnight_change,humidity_temp_3pm_interaction


**Interpretation:** The overlap between the manual and selected lists shows that the feature engineering remains grounded in meteorological logic, while the differences show where the aligned workflow adds value. In particular, the retained spatial variables suggest that rainfall prediction benefits from representing where the atmosphere is being observed, not only what is being measured locally.

## Step 9: Save the modeling-ready artifacts

The final task of this notebook is to preserve the cleaned dataset, the ranking table, and the selected shortlist in a reusable form. This is less glamorous than model training, but it is essential for reproducibility: the next notebook should inherit a fixed candidate table rather than rebuild it differently each time.

In [9]:
OUT_DATA = Path('../data/preprocessed/rain_model_dataset_aligned.csv')
OUT_RANKING = Path('../data/preprocessed/ines_feature_ranking_aligned.csv')
OUT_FEATURES = Path('../data/preprocessed/ines_selected_top25_features_aligned.txt')

OUT_DATA.parent.mkdir(parents=True, exist_ok=True)

df_model_candidates.to_csv(OUT_DATA, index=False)
feature_ranking.to_csv(OUT_RANKING, index=False)
OUT_FEATURES.write_text('\n'.join(selected_top_25), encoding='utf-8')

print('Saved aligned candidate modelling dataset to:', OUT_DATA.resolve())
print('Saved aligned feature ranking to:', OUT_RANKING.resolve())
print('Saved aligned top-25 feature list to:', OUT_FEATURES.resolve())


Saved aligned candidate modelling dataset to: C:\Users\user 1\Desktop\feb26_bds_int_weather\data\processed\rain_model_dataset_aligned.csv
Saved aligned feature ranking to: C:\Users\user 1\Desktop\feb26_bds_int_weather\data\processed\ines_feature_ranking_aligned.csv
Saved aligned top-25 feature list to: C:\Users\user 1\Desktop\feb26_bds_int_weather\data\processed\ines_selected_top25_features_aligned.txt


**Interpretation:** The preprocessing stage now ends with a coherent transition to modeling. The saved artifacts mean the model-comparison notebook can focus on algorithmic choices and probability quality, while the lineage of the data preparation remains explicit and reproducible. The deployment claim is also clearer once the pipeline is described honestly: the strongest results apply to forecasting within the observed station network, since location remains an active predictor and some repairs use same-date cross-station context before train-fit regime fallbacks take over.

### Transition to Notebook 3

The project now has everything needed for a serious modeling comparison: a cleaned supervised dataset, engineered predictors that reflect rainfall mechanisms, and a shortlist selected without using the final test block. The next step is to test which model family turns that information into the most credible probability of rain tomorrow. For the aligned top-25 surface, the resulting validation numbers should be treated as exploratory screening evidence rather than as a fully nested performance estimate.